# Treino do classificador leve para o ESP32 (classifier_head.hpp)

O modelo AST (`modelo.ipynb`) tem ~86M de parâmetros e não cabe em um ESP32 (flash de poucos MB, RAM de algumas centenas de KB, sem runtime de transformer). O firmware já extrai features leves de áudio (RMS, RMS em dB, centroide espectral e 13 MFCCs — 16 valores no total) em `features.cpp`, e esperava um classificador compatível, mas o `classifier_head.hpp` anterior expandia essas 16 features para 1024 dimensões de forma artificial (senoide, sem relação com o treino), gerando um classificador que não funciona.

Este notebook treina um MLP pequeno (entrada de 16 features -> 2 classes) do zero, usando o dataset público **ESC-50** (classe `dog_bark`), reimplementando em Python a mesma extração de features do firmware para garantir paridade, e exporta os pesos treinados para um novo `classifier_head.hpp` compatível com `sketch.ino`.

## Instalação das dependências

In [ ]:
!pip install -q \
    torch \
    librosa \
    soundfile \
    numpy \
    scikit-learn \
    pandas \
    onnx \
    onnxruntime

## Download do dataset ESC-50

O [ESC-50](https://github.com/karolpiczak/ESC-50) é um dataset público com 2000 clipes de 5s rotulados em 50 classes. A classe `dog_bark` (categoria "animals") é usada como exemplo positivo (`latido`); as demais 49 classes (chuva, motor, tosse, etc.) são usadas como exemplo negativo (`sem_latido`), dando um classificador robusto a ruído ambiente em geral, não só "qualquer coisa que não seja cachorro".

In [2]:
import os

DATASET_DIR = "/content/ESC-50"

if not os.path.isdir(DATASET_DIR):
    !git clone --depth 1 https://github.com/karolpiczak/ESC-50.git {DATASET_DIR}

AUDIO_DIR = os.path.join(DATASET_DIR, "audio")
META_PATH = os.path.join(DATASET_DIR, "meta", "esc50.csv")

print("Áudio em:", AUDIO_DIR)
print("Metadados em:", META_PATH)

Áudio em: /content/ESC-50/audio
Metadados em: /content/ESC-50/meta/esc50.csv


In [3]:
import pandas as pd

metadata = pd.read_csv(META_PATH)

metadata["rotulo"] = (metadata["category"] == "dog").astype(int)

print("Total de arquivos:", len(metadata))
print("Latidos (dog_bark):", int(metadata["rotulo"].sum()))
print("Não-latidos:", int((metadata["rotulo"] == 0).sum()))

metadata.head()

Total de arquivos: 2000
Latidos (dog_bark): 40
Não-latidos: 1960


,filename,fold,target,category,esc10,src_file,take,rotulo
0,1-100032-A-0.wav,1,0,dog,True,100032,A,1
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A,0
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A,0
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B,0
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A,0


## Extração de features (paridade com `features.cpp`)

Para o modelo treinado em Python se comportar da mesma forma no ESP32, a extração de features aqui replica exatamente a lógica de `src/sketch/features.cpp`:

- Janela de **512 amostras** a 16 kHz, remoção de DC, pré-ênfase (0.97), janela de Hamming.
- FFT, espectro de potência, **centroide espectral**.
- RMS e RMS em dB.
- **26 filtros mel** triangulares (300 Hz a Nyquist) + **13 coeficientes MFCC** via DCT-II (sem normalização, igual ao C++).

Não usamos `librosa.feature.mfcc` diretamente porque ele usa parâmetros (nº de filtros, faixa de frequência, normalização da DCT) diferentes dos do firmware — a rede precisa aprender sobre a mesma distribuição de features que o ESP32 vai calcular em produção.

In [4]:
import numpy as np
import librosa

TAXA_AMOSTRAGEM = 16000
TAMANHO_JANELA = 512
PASSO_JANELA = 160
QUANTIDADE_MFCC = 13
QUANTIDADE_FILTROS_MEL = 26
EPSILON = 1.0e-12


def hz_para_mel(frequencia):
    return 1127.0 * np.log(1.0 + frequencia / 700.0)


def mel_para_hz(mel):
    return 700.0 * (np.exp(mel / 1127.0) - 1.0)


def extrair_features_janela(amostras, taxa_amostragem=TAXA_AMOSTRAGEM):
    """Replica FeatureExtractor::calcular de features.cpp para uma janela de TAMANHO_JANELA amostras int16."""
    amostras = amostras.astype(np.float64)

    media = amostras.mean()
    centralizado = amostras - media
    energia = np.sum(centralizado ** 2)

    rms = np.sqrt(energia / TAMANHO_JANELA) / 32768.0
    rms_db = 20.0 * np.log10(rms + EPSILON)

    pre_enfase = np.empty(TAMANHO_JANELA)
    pre_enfase[0] = centralizado[0]
    pre_enfase[1:] = centralizado[1:] - 0.97 * centralizado[:-1]

    janela_hamming = 0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(TAMANHO_JANELA) / (TAMANHO_JANELA - 1))
    sinal_janelado = pre_enfase * janela_hamming

    espectro = np.fft.fft(sinal_janelado, n=TAMANHO_JANELA)
    metade = TAMANHO_JANELA // 2
    magnitude = np.abs(espectro[:metade])
    magnitude[0] = 0.0
    potencia = magnitude ** 2

    frequencias = np.arange(metade) * taxa_amostragem / TAMANHO_JANELA
    soma_magnitude = magnitude.sum()
    soma_frequencia = np.sum(frequencias * magnitude)
    centroide_espectral = soma_frequencia / soma_magnitude if soma_magnitude > EPSILON else 0.0

    mel_minimo = hz_para_mel(300.0)
    mel_maximo = hz_para_mel(taxa_amostragem / 2.0)
    pontos_mel = mel_minimo + (mel_maximo - mel_minimo) * np.arange(QUANTIDADE_FILTROS_MEL + 2) / (QUANTIDADE_FILTROS_MEL + 1)
    pontos_hz = mel_para_hz(pontos_mel)
    pontos_bin = np.floor((TAMANHO_JANELA + 1) * pontos_hz / taxa_amostragem).astype(int)
    pontos_bin = np.clip(pontos_bin, 0, metade - 1)

    energia_mel = np.zeros(QUANTIDADE_FILTROS_MEL)
    for filtro in range(QUANTIDADE_FILTROS_MEL):
        inicio, meio, fim = pontos_bin[filtro], pontos_bin[filtro + 1], pontos_bin[filtro + 2]
        for b in range(inicio, meio):
            if meio > inicio:
                energia_mel[filtro] += potencia[b] * (b - inicio) / (meio - inicio)
        for b in range(meio, min(fim + 1, metade)):
            if fim > meio:
                energia_mel[filtro] += potencia[b] * (fim - b) / (fim - meio)

    log_energia_mel = np.log(energia_mel + EPSILON)
    mfcc = np.zeros(QUANTIDADE_MFCC)
    for coeficiente in range(QUANTIDADE_MFCC):
        indices_filtro = np.arange(QUANTIDADE_FILTROS_MEL)
        mfcc[coeficiente] = np.sum(
            log_energia_mel * np.cos(np.pi * coeficiente * (indices_filtro + 0.5) / QUANTIDADE_FILTROS_MEL)
        )

    return {
        "rms": float(rms),
        "rms_db": float(rms_db),
        "centroide_espectral": float(centroide_espectral),
        "mfcc": mfcc.astype(np.float32),
    }


def vetor_features(features):
    return np.concatenate(
        [[features["rms"], features["rms_db"], features["centroide_espectral"]], features["mfcc"]]
    ).astype(np.float32)


def extrair_janelas_audio(caminho_audio, taxa_amostragem=TAXA_AMOSTRAGEM):
    """Carrega um áudio e retorna o vetor de 16 features (rms, rms_db, centroide, 13 mfcc) de cada janela deslizante."""
    audio, sr = librosa.load(caminho_audio, sr=taxa_amostragem, mono=True)
    amostras_int16 = np.clip(audio * 32768.0, -32768, 32767).astype(np.int16)

    vetores = []
    inicio = 0
    while inicio + TAMANHO_JANELA <= len(amostras_int16):
        janela = amostras_int16[inicio:inicio + TAMANHO_JANELA]
        features = extrair_features_janela(janela, taxa_amostragem)
        vetores.append(vetor_features(features))
        inicio += PASSO_JANELA

    return vetores

## Construção do dataset de treino

O classificador no ESP32 roda **por janela** (512 amostras, passo de 160), não por clipe inteiro. Por isso, em vez de calcular a média das features de um clipe de 5s (o que diluiria um latido curto dentro de silêncio), cada janela de cada áudio vira um exemplo de treino independente, herdando o rótulo do clipe. Isso também gera muito mais exemplos de treino a partir dos 2000 clipes do ESC-50.

In [7]:
import random
import os

random.seed(42)
np.random.seed(42)

DATASET_DIR = "/content/ESC-50"
AUDIO_DIR = os.path.join(DATASET_DIR, "audio")
META_PATH = os.path.join(DATASET_DIR, "meta", "esc50.csv")

if not os.path.exists(META_PATH):
    raise FileNotFoundError(
        f"Dataset ESC-50 não encontrado em {DATASET_DIR}. "
        "Execute a célula de download antes desta e confirme que o CSV foi baixado."
    )

metadata = pd.read_csv(META_PATH)
if "category" not in metadata.columns:
    if "label" in metadata.columns:
        metadata = metadata.rename(columns={"label": "category"})
    else:
        raise ValueError(f"Nenhuma coluna de categoria encontrada em {META_PATH}. Colunas disponíveis: {list(metadata.columns)}")

metadata = metadata.dropna(subset=["filename"]).copy()
metadata["category"] = metadata["category"].fillna("").astype(str)
category_lower = metadata["category"].str.lower()

# O ESC-50 usa 'dog_bark' em muitas versões, mas algumas exportações usam variações como 'dog bark' ou 'bark'.
mask_bark = (
    category_lower.str.contains("dog", regex=False)
)

metadata["rotulo"] = mask_bark.astype(int)

print("Categorias presentes:", sorted(metadata["category"].dropna().astype(str).str.lower().unique())[:20])
print("Total de arquivos do dataset:", len(metadata))
print("Latidos detectados:", int(metadata["rotulo"].sum()))
print("Não-latidos:", int((metadata["rotulo"] == 0).sum()))

arquivos_bark = metadata[metadata["rotulo"] == 1]["filename"].tolist()
arquivos_outros = metadata[metadata["rotulo"] == 0]["filename"].tolist()

if len(arquivos_bark) == 0 or len(arquivos_outros) == 0:
    print("Valores de category: ", metadata["category"].astype(str).head(20).tolist())
    raise ValueError(
        "Dataset carregado, mas nenhuma categoria de 'dog_bark' foi reconhecida. "
        "Verifique o CSV do ESC-50, a coluna 'category' ou o download do dataset."
    )

JANELAS_POR_CLIPE = 40
arquivos_outros_amostrados = random.sample(arquivos_outros, k=min(len(arquivos_outros), len(arquivos_bark) * 5))

X, y = [], []

for filename in arquivos_bark:
    caminho = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(caminho):
        print(f"Arquivo ausente: {caminho}")
        continue
    janelas = extrair_janelas_audio(caminho)
    if not janelas:
        continue
    janelas_amostradas = random.sample(janelas, k=min(len(janelas), JANELAS_POR_CLIPE))
    X.extend(janelas_amostradas)
    y.extend([1] * len(janelas_amostradas))

for filename in arquivos_outros_amostrados:
    caminho = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(caminho):
        print(f"Arquivo ausente: {caminho}")
        continue
    janelas = extrair_janelas_audio(caminho)
    if not janelas:
        continue
    janelas_amostradas = random.sample(janelas, k=min(len(janelas), JANELAS_POR_CLIPE // 5))
    X.extend(janelas_amostradas)
    y.extend([0] * len(janelas_amostradas))

if len(X) == 0 or len(y) == 0:
    raise ValueError(
        "Nenhuma janela de áudio foi gerada após a leitura do dataset. "
        "Confirme se os arquivos de áudio do ESC-50 foram baixados corretamente e a estrutura do diretório está correta."
    )

X = np.stack(X)
y = np.array(y)

print("Exemplos totais:", len(X))
print("Latido:", int(y.sum()), "| Sem latido:", int((y == 0).sum()))
print("Shape de X:", X.shape)


Categorias presentes: ['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks']
Total de arquivos do dataset: 2000
Latidos detectados: 40
Não-latidos: 1960
Exemplos totais: 3200
Latido: 1600 | Sem latido: 1600
Shape de X: (3200, 16)


## Normalização e split treino/teste

As features têm escalas bem diferentes (RMS ~0-1, RMS dB ~-60 a 0, centroide em Hz, MFCC em unidades log). Normalizamos com z-score (média/desvio do conjunto de treino) e guardamos esses parâmetros para embutir no firmware, já que o ESP32 precisa aplicar a mesma normalização antes de rodar a rede.

In [8]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

media_features = X_treino.mean(axis=0)
desvio_features = X_treino.std(axis=0)
desvio_features[desvio_features < 1e-6] = 1e-6  # evita divisão por zero em features constantes

X_treino_norm = (X_treino - media_features) / desvio_features
X_teste_norm = (X_teste - media_features) / desvio_features

print("Treino:", X_treino_norm.shape, "| Teste:", X_teste_norm.shape)

Treino: (2560, 16) | Teste: (640, 16)


## Definição e treino do MLP

Arquitetura pequena o suficiente para caber tranquilamente na flash/RAM de um ESP32: `16 -> 32 -> 16 -> 2`, com ReLU nas camadas ocultas e softmax na saída (mesmo formato do `ClassifierHead::inferir` original).

In [9]:
import torch
import torch.nn as nn

INPUT_DIM = X.shape[1]  # 16
HIDDEN_1 = 32
HIDDEN_2 = 16
OUTPUT_DIM = 2

torch.manual_seed(42)


class BarkClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense1 = nn.Linear(INPUT_DIM, HIDDEN_1)
        self.dense2 = nn.Linear(HIDDEN_1, HIDDEN_2)
        self.saida = nn.Linear(HIDDEN_2, OUTPUT_DIM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.dense1(x))
        x = self.relu(self.dense2(x))
        return self.saida(x)  # logits (softmax aplicado na loss/inferência)


modelo_classificador = BarkClassifier()
print(modelo_classificador)

BarkClassifier(
  (dense1): Linear(in_features=16, out_features=32, bias=True)
  (dense2): Linear(in_features=32, out_features=16, bias=True)
  (saida): Linear(in_features=16, out_features=2, bias=True)
  (relu): ReLU()
)


In [10]:
from torch.utils.data import TensorDataset, DataLoader

X_treino_t = torch.tensor(X_treino_norm, dtype=torch.float32)
y_treino_t = torch.tensor(y_treino, dtype=torch.long)
X_teste_t = torch.tensor(X_teste_norm, dtype=torch.float32)
y_teste_t = torch.tensor(y_teste, dtype=torch.long)

loader_treino = DataLoader(TensorDataset(X_treino_t, y_treino_t), batch_size=64, shuffle=True)

criterio = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(modelo_classificador.parameters(), lr=1e-3)

EPOCAS = 50

for epoca in range(1, EPOCAS + 1):
    modelo_classificador.train()
    perda_total = 0.0
    for xb, yb in loader_treino:
        otimizador.zero_grad()
        logits = modelo_classificador(xb)
        perda = criterio(logits, yb)
        perda.backward()
        otimizador.step()
        perda_total += perda.item() * xb.size(0)

    if epoca % 5 == 0 or epoca == 1:
        modelo_classificador.eval()
        with torch.no_grad():
            logits_teste = modelo_classificador(X_teste_t)
            acuracia = (logits_teste.argmax(dim=1) == y_teste_t).float().mean().item()
        print(f"Época {epoca:3d} | perda treino: {perda_total / len(X_treino_t):.4f} | acurácia teste: {acuracia:.4f}")

Época   1 | perda treino: 0.6789 | acurácia teste: 0.6891
Época   5 | perda treino: 0.4782 | acurácia teste: 0.7203
Época  10 | perda treino: 0.4106 | acurácia teste: 0.7734
Época  15 | perda treino: 0.3761 | acurácia teste: 0.7875
Época  20 | perda treino: 0.3555 | acurácia teste: 0.7906
Época  25 | perda treino: 0.3405 | acurácia teste: 0.7812
Época  30 | perda treino: 0.3300 | acurácia teste: 0.7859
Época  35 | perda treino: 0.3172 | acurácia teste: 0.7875
Época  40 | perda treino: 0.3086 | acurácia teste: 0.7969
Época  45 | perda treino: 0.2983 | acurácia teste: 0.7937
Época  50 | perda treino: 0.2899 | acurácia teste: 0.8031


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

modelo_classificador.eval()
with torch.no_grad():
    predicoes = modelo_classificador(X_teste_t).argmax(dim=1).numpy()

print(classification_report(y_teste, predicoes, target_names=["sem_latido", "latido"]))
print("Matriz de confusão:")
print(confusion_matrix(y_teste, predicoes))

              precision    recall  f1-score   support

  sem_latido       0.89      0.69      0.78       320
      latido       0.75      0.91      0.82       320

    accuracy                           0.80       640
   macro avg       0.82      0.80      0.80       640
weighted avg       0.82      0.80      0.80       640

Matriz de confusão:
[[222  98]
 [ 28 292]]


# Exportação para `classifier_head.hpp`

Gera o novo header C++ com:
- As dimensões reais (`CLASSIFIER_INPUT_DIM = 16`, sem a expansão artificial para 1024).
- Os parâmetros de normalização (`media`/`desvio`) aprendidos no treino, aplicados às features antes da rede — o ESP32 precisa fazer a mesma normalização.
- Os pesos/bias treinados das 3 camadas densas.
- `ClassifierHead::inferir` recebendo `AudioFeatures` diretamente (sem mais `embeddingFromFeatures` artificial).

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = "/content/bark_classifier.onnx"
dummy_input = torch.randn(1, INPUT_DIM, dtype=torch.float32)

modelo_classificador.eval()

torch.onnx.export(
    modelo_classificador,
    dummy_input,
    ONNX_PATH,
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={"features": {0: "batch_size"}},
    opset_version=17,
)

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

sessao_onnx = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
saida_onnx = sessao_onnx.run(None, {"features": dummy_input.numpy()})

saida_torch = modelo_classificador(dummy_input).detach().numpy()
max_diff = np.max(np.abs(saida_torch - saida_onnx[0]))

print("ONNX exportado em:", ONNX_PATH)
print("Saída ONNX shape:", saida_onnx[0].shape)
print("Diferença máxima PyTorch vs ONNX:", max_diff)

## Exportação para `classifier_head.hpp`

Gera o novo header C++ com:
- As dimensões reais (`CLASSIFIER_INPUT_DIM = 16`, sem a expansão artificial para 1024).
- Os parâmetros de normalização (`media`/`desvio`) aprendidos no treino, aplicados às features antes da rede — o ESP32 precisa fazer a mesma normalização.
- Os pesos/bias treinados das 3 camadas densas.
- `ClassifierHead::inferir` recebendo `AudioFeatures` diretamente (sem mais `embeddingFromFeatures` artificial).

In [12]:
def formatar_array_cpp(valores, casas_decimais=9):
    return ", ".join(f"{v:.{casas_decimais}g}f" for v in valores)


pesos = modelo_classificador.state_dict()

dense1_w = pesos["dense1.weight"].numpy()  # [HIDDEN_1, INPUT_DIM]
dense1_b = pesos["dense1.bias"].numpy()
dense2_w = pesos["dense2.weight"].numpy()  # [HIDDEN_2, HIDDEN_1]
dense2_b = pesos["dense2.bias"].numpy()
saida_w = pesos["saida.weight"].numpy()  # [OUTPUT_DIM, HIDDEN_2]
saida_b = pesos["saida.bias"].numpy()

# Exportado em layout [entrada][saida] (transposto), para casar com o loop
# `soma += entrada[i] * PESOS[i * DIM_SAIDA + j]` usado em classifier_head.hpp.
dense1_w_t = dense1_w.T.flatten()  # INPUT_DIM * HIDDEN_1
dense2_w_t = dense2_w.T.flatten()  # HIDDEN_1 * HIDDEN_2
saida_w_t = saida_w.T.flatten()    # HIDDEN_2 * OUTPUT_DIM

print("dense1_w:", dense1_w_t.shape, "dense2_w:", dense2_w_t.shape, "saida_w:", saida_w_t.shape)

dense1_w: (512,) dense2_w: (512,) saida_w: (32,)


In [13]:
CLASSIFIER_THRESHOLD = 0.5

header = f"""#pragma once

#include <Arduino.h>
#include <math.h>

#include "features.hpp"

constexpr size_t CLASSIFIER_INPUT_DIM = {INPUT_DIM};
constexpr size_t CLASSIFIER_HIDDEN_1 = {HIDDEN_1};
constexpr size_t CLASSIFIER_HIDDEN_2 = {HIDDEN_2};
constexpr size_t CLASSIFIER_OUTPUT_DIM = {OUTPUT_DIM};
constexpr float CLASSIFIER_THRESHOLD = {CLASSIFIER_THRESHOLD}f;

// Parâmetros de normalização (z-score) aprendidos no treino, na mesma ordem de AudioFeatures
// (rms, rmsDb, centroideEspectral, mfcc[0..12]).
constexpr float K_CLASSIFIER_FEATURE_MEAN[CLASSIFIER_INPUT_DIM] = {{ {formatar_array_cpp(media_features)} }};
constexpr float K_CLASSIFIER_FEATURE_STD[CLASSIFIER_INPUT_DIM] = {{ {formatar_array_cpp(desvio_features)} }};

constexpr float K_CLASSIFIER_DENSE_1_WEIGHTS[CLASSIFIER_INPUT_DIM * CLASSIFIER_HIDDEN_1] = {{ {formatar_array_cpp(dense1_w_t)} }};
constexpr float K_CLASSIFIER_DENSE_1_BIAS[CLASSIFIER_HIDDEN_1] = {{ {formatar_array_cpp(dense1_b)} }};
constexpr float K_CLASSIFIER_DENSE_2_WEIGHTS[CLASSIFIER_HIDDEN_1 * CLASSIFIER_HIDDEN_2] = {{ {formatar_array_cpp(dense2_w_t)} }};
constexpr float K_CLASSIFIER_DENSE_2_BIAS[CLASSIFIER_HIDDEN_2] = {{ {formatar_array_cpp(dense2_b)} }};
constexpr float K_CLASSIFIER_OUTPUT_WEIGHTS[CLASSIFIER_HIDDEN_2 * CLASSIFIER_OUTPUT_DIM] = {{ {formatar_array_cpp(saida_w_t)} }};
constexpr float K_CLASSIFIER_OUTPUT_BIAS[CLASSIFIER_OUTPUT_DIM] = {{ {formatar_array_cpp(saida_b)} }};

inline void softmax(const float *logits, float *probs, size_t quantidade) {{
    float maximo = -INFINITY;
    for (size_t i = 0; i < quantidade; ++i) {{
        if (logits[i] > maximo) {{
            maximo = logits[i];
        }}
    }}
    float soma = 0.0f;
    for (size_t i = 0; i < quantidade; ++i) {{
        const float valor = expf(logits[i] - maximo);
        probs[i] = valor;
        soma += valor;
    }}
    for (size_t i = 0; i < quantidade; ++i) {{
        probs[i] /= soma;
    }}
}}

class ClassifierHead {{
public:
    static void embeddingFromFeatures(const AudioFeatures &features, float *embedding) {{
        const float bruto[CLASSIFIER_INPUT_DIM] = {{
            features.rms,
            features.rmsDb,
            features.centroideEspectral,
            features.mfcc[0], features.mfcc[1], features.mfcc[2], features.mfcc[3],
            features.mfcc[4], features.mfcc[5], features.mfcc[6], features.mfcc[7],
            features.mfcc[8], features.mfcc[9], features.mfcc[10], features.mfcc[11],
            features.mfcc[12],
        }};
        for (size_t i = 0; i < CLASSIFIER_INPUT_DIM; ++i) {{
            embedding[i] = (bruto[i] - K_CLASSIFIER_FEATURE_MEAN[i]) / K_CLASSIFIER_FEATURE_STD[i];
        }}
    }}

    static void inferir(const float *embedding, float *probs) {{
        float hidden1[CLASSIFIER_HIDDEN_1] = {{}};
        for (size_t j = 0; j < CLASSIFIER_HIDDEN_1; ++j) {{
            float soma = K_CLASSIFIER_DENSE_1_BIAS[j];
            for (size_t i = 0; i < CLASSIFIER_INPUT_DIM; ++i) {{
                soma += embedding[i] * K_CLASSIFIER_DENSE_1_WEIGHTS[i * CLASSIFIER_HIDDEN_1 + j];
            }}
            hidden1[j] = soma > 0.0f ? soma : 0.0f;
        }}

        float hidden2[CLASSIFIER_HIDDEN_2] = {{}};
        for (size_t j = 0; j < CLASSIFIER_HIDDEN_2; ++j) {{
            float soma = K_CLASSIFIER_DENSE_2_BIAS[j];
            for (size_t i = 0; i < CLASSIFIER_HIDDEN_1; ++i) {{
                soma += hidden1[i] * K_CLASSIFIER_DENSE_2_WEIGHTS[i * CLASSIFIER_HIDDEN_2 + j];
            }}
            hidden2[j] = soma > 0.0f ? soma : 0.0f;
        }}

        float logits[CLASSIFIER_OUTPUT_DIM] = {{}};
        for (size_t j = 0; j < CLASSIFIER_OUTPUT_DIM; ++j) {{
            float soma = K_CLASSIFIER_OUTPUT_BIAS[j];
            for (size_t i = 0; i < CLASSIFIER_HIDDEN_2; ++i) {{
                soma += hidden2[i] * K_CLASSIFIER_OUTPUT_WEIGHTS[i * CLASSIFIER_OUTPUT_DIM + j];
            }}
            logits[j] = soma;
        }}

        softmax(logits, probs, CLASSIFIER_OUTPUT_DIM);
    }}

    static void inferir(const AudioFeatures &features, float *probs) {{
        float embedding[CLASSIFIER_INPUT_DIM] = {{}};
        embeddingFromFeatures(features, embedding);
        inferir(embedding, probs);
    }}
}};
"""

OUTPUT_PATH = "/content/classifier_head.hpp"
with open(OUTPUT_PATH, "w") as f:
    f.write(header)

print("Gerado:", OUTPUT_PATH)
print("Tamanho:", os.path.getsize(OUTPUT_PATH) / 1024, "KB")

Gerado: /content/classifier_head.hpp
Tamanho: 20.07421875 KB


In [14]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    print(f"Não está no Colab — copie o arquivo gerado em {OUTPUT_PATH} para src/sketch/classifier_head.hpp manualmente.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Próximos passos

1. Substituir `src/sketch/classifier_head.hpp` pelo arquivo gerado.
2. Recompilar e gravar o firmware no ESP32.
3. Validar em campo: o `CLASSIFIER_THRESHOLD` (0.5) pode ser ajustado conforme a taxa de falsos positivos/negativos observada, sem precisar retreinar.
4. Se a acurácia em teste (`classification_report` acima) não for satisfatória, aumentar `JANELAS_POR_CLIPE`, adicionar mais dados (outros datasets de latido) ou aumentar `HIDDEN_1`/`HIDDEN_2` — mantendo em mente o orçamento de RAM/flash do ESP32.